# End-to-End Fine-Tuning: Tiny Offline GPT-2

This notebook demonstrates a full, original fine-tuning pipeline using an open-source GPT-2 style architecture without downloading any external model files.

## 1) Install dependencies
Run this once if your environment is fresh.

In [1]:
%pip install -q transformers datasets accelerate torch tokenizers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
import numpy as np
import torch
from datasets import Dataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import (
    DataCollatorForLanguageModeling,
    GPT2Config,
    GPT2LMHeadModel,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


Using device: cpu


## 2) Build a small custom dataset

In [3]:
examples = [
    {'instruction': 'Write a one-line Python function that squares a number.', 'response': 'def square(x): return x * x'},
    {'instruction': 'Explain what overfitting means in machine learning.', 'response': 'Overfitting happens when a model memorizes training data patterns and performs poorly on new data.'},
    {'instruction': 'Give three tips for debugging a neural network training loop.', 'response': '1) Verify tensor shapes. 2) Overfit on a tiny batch first. 3) Track loss and gradients each step.'},
    {'instruction': 'Summarize why validation sets are important.', 'response': 'Validation sets estimate generalization and help tune hyperparameters without leaking test information.'},
    {'instruction': 'Write a short note on reproducibility in AI experiments.', 'response': 'Reproducibility requires fixed seeds, tracked configs, versioned data, and logged environments.'},
    {'instruction': 'Describe gradient descent in plain language.', 'response': 'Gradient descent iteratively updates model parameters in the direction that reduces prediction error.'},
    {'instruction': 'What is the difference between training loss and validation loss?', 'response': 'Training loss is measured on seen data, while validation loss is measured on held-out data to check generalization.'},
    {'instruction': 'Write one sentence about transfer learning.', 'response': 'Transfer learning starts from a pretrained model and adapts it to a new task with less data and time.'},
]

def to_prompt(item):
    return f"### Instruction:\n{item['instruction']}\n\n### Response:\n{item['response']}"

texts = [to_prompt(item) for item in examples]
dataset = Dataset.from_dict({'text': texts})
split_dataset = dataset.train_test_split(test_size=0.25, seed=SEED)
split_dataset


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 6
    })
    test: Dataset({
        features: ['text'],
        num_rows: 2
    })
})

## 3) Train a tokenizer on the local dataset (offline)

In [4]:
raw_tokenizer = Tokenizer(models.WordLevel(unk_token='[UNK]'))
raw_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.WordLevelTrainer(special_tokens=['[PAD]', '[UNK]', '[BOS]', '[EOS]'])
raw_tokenizer.train_from_iterator(texts, trainer=trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
    bos_token='[BOS]',
    eos_token='[EOS]',
    unk_token='[UNK]',
    pad_token='[PAD]',
)
tokenizer.vocab_size


/home/runner/.local/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


141

## 4) Tokenize

In [5]:
max_length = 128
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=max_length)

tokenized = split_dataset.map(tokenize, batched=True, remove_columns=['text'])
tokenized


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 6
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2
    })
})

## 5) Build a tiny GPT-2 style model and train

In [6]:
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=128,
    n_ctx=128,
    n_embd=128,
    n_layer=2,
    n_head=4,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

model = GPT2LMHeadModel(config)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='outputs/tiny-gpt2-offline-finetuned',
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=1,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=collator,
)

trainer.train()


/home/runner/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,4.881200,4.830234
2,4.653600,4.745075
3,4.511800,4.700516
4,4.455000,4.678293
5,4.406800,4.669156


/home/runner/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/home/runner/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/home/runner/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/home/runner/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/home/runner/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=15, training_loss=4.612345059712728, metrics={'train_runtime': 0.2957, 'train_samples_per_second': 101.445, 'train_steps_per_second': 50.722, 'total_flos': 9142272000.0, 'train_loss': 4.612345059712728, 'epoch': 5.0})

## 6) Save model + tokenizer

In [7]:
trainer.save_model('outputs/tiny-gpt2-offline-finetuned/final')
tokenizer.save_pretrained('outputs/tiny-gpt2-offline-finetuned/final')


('outputs/tiny-gpt2-offline-finetuned/final/tokenizer_config.json',
 'outputs/tiny-gpt2-offline-finetuned/final/special_tokens_map.json',
 'outputs/tiny-gpt2-offline-finetuned/final/tokenizer.json')

## 7) Quick generation check

In [8]:
prompt = '### Instruction:\nExplain transfer learning in one sentence.\n\n### Response:\n'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    output_tokens = model.generate(**inputs, max_new_tokens=30, do_sample=True, top_p=0.95, temperature=0.9)
print(tokenizer.decode(output_tokens[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:3 for open-end generation.


### Instruction : Explain transfer learning in one sentence. ### Response : between between squares a - one from estimate tensor Summarize debugging prediction function language Instruction reduces. Give :. learning line sets * Track validation what, in x
